## 경로 설정 (레포 공통)

이 셀은 레포 어디에서 노트북을 열어도 `data/` 폴더를 자동으로 찾아 **상대경로 기반**으로 경로를 지정한다.
- `data/raw` 원자료 → `data/interim` 중간 자료 → `data/processed` 최종본, 그림·표는 `outputs/`에 저장한다.
- 노트북은 레포 안 어느 위치에서 실행해도 되지만, 레포 폴더 구조(`data/`, `src/`, `outputs/`)를 바꾸지 않아야 한다.

In [1]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "data" / "raw").exists())
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
FIG_DIR = ROOT / "outputs" / "figures"
TAB_DIR = ROOT / "outputs" / "tables"
for _d in (INTERIM, PROCESSED, FIG_DIR, TAB_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print("프로젝트 루트 폴더:", ROOT.name)

프로젝트 루트 폴더: BAF-26-2-finance_2


# 수익성 원자료 검증

## 목적

이 노트북은 `수익성_전체통합_1994_2025.csv`가 **후속 분석에 사용 가능한 원자료인지 확인**하기 위한 최소한의 품질 검증을 수행한다.

원자료 단계에서는 업종별 장기 추세나 세부 분포를 깊게 해석하지 않는다.  
본격적인 EDA는 이후 `재무_원자재_병합패널.csv` 등 **최종 분석용 병합패널**에서 수행한다.

### 이번 노트북에서 확인하는 내용

1. 데이터 기본 구조
2. 패널 구조의 정상 여부
3. 결측·중복·무한대·0값·음수값
4. 적자 및 극단 관측치의 성격
5. 주요 수익성 지표의 내부 산식 일관성
6. 원자료 사용 가능 여부에 대한 최종 판단

## 1. 데이터 불러오기

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# financial_project/notebooks/ 아래에서 실행하는 것을 기준으로 함
file_path = INTERIM / "ISTANS 수익성" / "수익성_전체통합_1994_2025.csv"

df = pd.read_csv(file_path)

print(f"파일 존재 여부: {file_path.exists()}")
print(f"데이터 크기: {df.shape[0]:,}행 × {df.shape[1]}열")

파일 존재 여부: True
데이터 크기: 1,280행 × 8열


## 2. 기본 구조 확인

행·열 수, 변수명, 자료형, 결측 여부를 확인한다.  
이 단계의 목적은 **파일이 예상한 형태로 구성되어 있는지** 빠르게 점검하는 것이다.

In [3]:
basic_info = pd.DataFrame({
    "변수명": df.columns,
    "자료형": df.dtypes.astype(str).values,
    "결측개수": df.isna().sum().values,
    "결측비율(%)": (df.isna().mean() * 100).round(2).values,
    "고유값개수": df.nunique(dropna=True).values
})

display(basic_info)

print(f"업종 수: {df['업종'].nunique()}")
print(f"연도 수: {df['연도'].nunique()}")
print(f"기간: {df['연도'].min()} ~ {df['연도'].max()}")

,변수명,자료형,결측개수,결측비율(%),고유값개수
0,업종코드,str,0,0.0,40
1,업종,str,0,0.0,40
2,연도,int64,0,0.0,32
3,매출액,float64,0,0.0,1240
4,영업이익,float64,0,0.0,966
5,당기순이익,float64,0,0.0,898
6,영업이익률,float64,0,0.0,247
7,이자보상비율,float64,0,0.0,1204


업종 수: 40
연도 수: 32
기간: 1994 ~ 2025


### 확인 결과

- 총 40개 업종과 32개 연도가 존재한다.
- 1994~2025년 기간의 자료다.
- 모든 변수에서 결측치는 확인되지 않는다.
- 업종코드·업종·연도와 수익성 관련 수치형 변수의 자료형도 분석에 사용 가능한 형태다.

## 3. 패널 구조 검증

원자료가 **40개 업종 × 32개 연도**의 완전한 패널인지 확인한다.

확인 항목:

- 업종코드와 업종명이 1:1로 대응하는가
- 동일 업종-연도 조합이 중복되는가
- 누락된 업종-연도 조합이 있는가
- 모든 연도에 40개 업종이 존재하는가

In [4]:
# 업종코드 ↔ 업종명 1:1 대응 확인
code_name_problem = (
    df.groupby("업종코드")["업종"].nunique()
      .reset_index(name="업종명_개수")
      .query("업종명_개수 != 1")
)

name_code_problem = (
    df.groupby("업종")["업종코드"].nunique()
      .reset_index(name="업종코드_개수")
      .query("업종코드_개수 != 1")
)

# 업종 × 연도 중복 확인
duplicate_panel = df[
    df.duplicated(subset=["업종코드", "연도"], keep=False)
].copy()

# 기대되는 전체 업종 × 연도 조합 생성
all_industries = df[["업종코드", "업종"]].drop_duplicates()
all_years = pd.DataFrame({"연도": sorted(df["연도"].unique())})

expected_panel = (
    all_industries.assign(key=1)
    .merge(all_years.assign(key=1), on="key")
    .drop(columns="key")
)

actual_panel = df[["업종코드", "업종", "연도"]].drop_duplicates()

# 누락 업종-연도 조합
missing_panel = (
    expected_panel
    .merge(
        actual_panel,
        on=["업종코드", "업종", "연도"],
        how="left",
        indicator=True
    )
    .query('_merge == "left_only"')
    .drop(columns="_merge")
)

# 연도별 관측 업종 수
year_industry_count = (
    df.groupby("연도")["업종코드"]
      .nunique()
      .reset_index(name="관측업종수")
)

is_balanced = (
    len(code_name_problem) == 0
    and len(name_code_problem) == 0
    and len(duplicate_panel) == 0
    and len(missing_panel) == 0
    and year_industry_count["관측업종수"].eq(40).all()
)

panel_result = pd.DataFrame({
    "확인 항목": [
        "업종코드-업종명 불일치",
        "업종-연도 중복",
        "누락 업종-연도 조합",
        "균형패널 여부"
    ],
    "결과": [
        len(code_name_problem) + len(name_code_problem),
        len(duplicate_panel),
        len(missing_panel),
        "예" if is_balanced else "아니오"
    ]
})

display(panel_result)

,확인 항목,결과
0,업종코드-업종명 불일치,0
1,업종-연도 중복,0
2,누락 업종-연도 조합,0
3,균형패널 여부,예


### 확인 결과

- 업종코드와 업종명은 1:1로 대응한다.
- 동일 업종-연도 중복은 없다.
- 누락된 업종-연도 조합도 없다.
- 모든 연도에서 40개 업종이 존재한다.

따라서 본 자료는 **40개 업종을 32년간 관측한 완전 균형패널**이다.

## 4. 기본 품질 점검

결측치 외에도 무한대, 0값, 음수값이 존재하는지 확인한다.

수익성 자료에서는 `영업이익`, `당기순이익`, `영업이익률`, `이자보상비율`의 음수값이 실제 적자를 나타낼 수 있으므로  
**음수라는 이유만으로 오류로 판단하거나 삭제하지 않는다.**

In [5]:
value_cols = df.columns[3:]

quality_rows = []

for col in value_cols:
    s = df[col]

    quality_rows.append({
        "변수명": col,
        "결측": s.isna().sum(),
        "무한대": np.isinf(s).sum(),
        "0값": (s == 0).sum(),
        "0값비율(%)": round((s == 0).mean() * 100, 2),
        "음수값": (s < 0).sum(),
        "음수비율(%)": round((s < 0).mean() * 100, 2)
    })

quality_check = pd.DataFrame(quality_rows)

display(quality_check)

,변수명,결측,무한대,0값,0값비율(%),음수값,음수비율(%)
0,매출액,0,0,0,0.00,0,0.00
1,영업이익,0,0,2,0.16,51,3.98
2,당기순이익,0,0,9,0.70,151,11.80
3,영업이익률,0,0,0,0.00,52,4.06
4,이자보상비율,0,0,0,0.00,52,4.06


### 확인 결과

- 모든 변수에서 결측치와 무한대값은 확인되지 않는다.
- `매출액`은 전 관측치에서 양수다.
- `영업이익`, `당기순이익`, `영업이익률`, `이자보상비율`에는 음수값이 존재한다.
- 이러한 음수값은 적자 또는 영업손실을 나타낼 수 있으므로 원자료 단계에서는 그대로 보존한다.
- 일부 이익 변수의 0값 역시 실제 값 또는 원자료의 표시 단위에서 발생한 반올림 값일 수 있으므로 임의로 처리하지 않는다.

## 5. 적자 및 주요 극단 관측치 확인

수익성 자료에서는 음수값 자체가 오류라기보다 중요한 경제적 상태일 수 있다.

따라서 다음을 확인한다.

- 영업이익이 음수인 관측치 수
- 당기순이익이 음수인 관측치 수
- 영업이익률이 음수인 관측치 수
- 이자보상비율이 음수인 관측치 수
- 각 수익성 지표의 상·하위 극단 관측치

In [6]:
loss_summary = pd.DataFrame({
    "항목": [
        "영업이익 < 0",
        "당기순이익 < 0",
        "영업이익률 < 0",
        "이자보상비율 < 0"
    ],
    "관측치수": [
        (df["영업이익"] < 0).sum(),
        (df["당기순이익"] < 0).sum(),
        (df["영업이익률"] < 0).sum(),
        (df["이자보상비율"] < 0).sum()
    ]
})

loss_summary["비율(%)"] = (
    loss_summary["관측치수"] / len(df) * 100
).round(2)

display(loss_summary)

print("\n[영업이익률 하위 10개]")
display(
    df.nsmallest(10, "영업이익률")[
        ["업종코드", "업종", "연도", "매출액", "영업이익", "영업이익률"]
    ].reset_index(drop=True)
)

print("\n[이자보상비율 하위 10개]")
display(
    df.nsmallest(10, "이자보상비율")[
        ["업종코드", "업종", "연도", "영업이익", "이자보상비율"]
    ].reset_index(drop=True)
)

print("\n[이자보상비율 상위 10개]")
display(
    df.nlargest(10, "이자보상비율")[
        ["업종코드", "업종", "연도", "영업이익", "이자보상비율"]
    ].reset_index(drop=True)
)

,항목,관측치수,비율(%)
0,영업이익 < 0,51,3.98
1,당기순이익 < 0,151,11.80
2,영업이익률 < 0,52,4.06
3,이자보상비율 < 0,52,4.06



[영업이익률 하위 10개]


,업종코드,업종,연도,매출액,영업이익,영업이익률
0,I3102,반도체,2001,7470.0,-1250.0,-16.7
1,I3208,철도,2000,435.0,-66.0,-15.2
2,I3312,조선,2021,32808.0,-4627.0,-14.1
3,I3208,철도,2019,2799.0,-346.0,-12.3
4,I3102,반도체,2008,14809.0,-1769.0,-11.9
5,I3204,전기기기,1999,6420.0,-723.0,-11.3
6,I3102,반도체,2002,6791.0,-683.0,-10.1
7,I3108,전지,2007,4463.0,-440.0,-9.9
8,I3312,조선,2015,43093.0,-4245.0,-9.9
9,I3103,디스플레이,2001,2850.0,-248.0,-8.7



[이자보상비율 하위 10개]


,업종코드,업종,연도,영업이익,이자보상비율
0,I3108,전지,2016,-710.0,-2655.1
1,I3108,전지,2007,-440.0,-2464.0
2,I3105,통신기기,2023,-10653.0,-963.0
3,I3104,컴퓨터,2019,-181.0,-930.0
4,I3312,조선,2021,-4627.0,-737.2
5,I3103,디스플레이,2019,-1389.0,-698.9
6,I3208,철도,2019,-346.0,-639.7
7,I3108,전지,2015,-142.0,-629.3
8,I3103,디스플레이,2011,-920.0,-526.1
9,I3301,석유정제,2020,-3423.0,-503.5



[이자보상비율 상위 10개]


,업종코드,업종,연도,영업이익,이자보상비율
0,I3402,담배,1994,205.0,104013.8
1,I3402,담배,2000,432.0,83279.4
2,I3402,담배,1999,267.0,50239.3
3,I3402,담배,1995,255.0,42448.4
4,I3402,담배,2016,1433.0,30658.3
5,I3402,담배,2015,1520.0,28984.0
6,I3402,담배,1996,237.0,24043.5
7,I3402,담배,2009,1119.0,17543.1
8,I3402,담배,1997,217.0,17480.1
9,I3402,담배,2014,1260.0,16214.4


### 해석

적자 관측치는 수익성 자료에서 자연스럽게 발생할 수 있으므로 단순 오류로 보지 않는다.

특히 `영업이익률`과 `이자보상비율`의 음수 여부가 거의 동일하게 움직이는 것은  
영업손실이 발생한 경우 이자보상 능력도 음수로 나타나는 구조와 일치한다.

반면 `이자보상비율`에는 매우 큰 양의 값도 존재한다.  
이 변수는 일반적으로 영업이익과 이자비용의 비율로 계산되기 때문에 이자비용이 매우 작으면 값이 크게 증가할 수 있다.

다만 현재 수익성 파일에는 `이자비용` 변수가 없으므로  
**이자보상비율의 산식을 이 파일만으로 독립적으로 재계산할 수 없다.**

따라서 극단값을 임의로 제거하지 않고 원자료에 유지하며,  
최종 분석용 병합패널에서 실제 분석기간에 포함되는 값만 다시 확인한다.

## 5-1. 음수값의 외부 근거 확인

수익성 변수의 음수값은 단순히 `이상치`로 처리하지 않고,  
**같은 시기·산업에서 실제 업황 악화 또는 수익성 저하가 있었는지 공식 자료를 통해 확인**한다.

먼저 음수 관측치가 어느 시기와 업종에 집중되어 있는지 요약한다.

In [7]:
negative_summary = []

for col in ["영업이익", "당기순이익", "영업이익률", "이자보상비율"]:
    tmp = df[df[col] < 0]

    negative_summary.append({
        "변수": col,
        "음수 관측치 수": len(tmp),
        "음수 비율(%)": round(len(tmp) / len(df) * 100, 2),
        "관련 업종 수": tmp["업종"].nunique(),
        "최초 연도": tmp["연도"].min(),
        "최종 연도": tmp["연도"].max()
    })

display(pd.DataFrame(negative_summary))

print("\n[연도별 영업이익률 음수 관측치 수]")
display(
    df[df["영업이익률"] < 0]
      .groupby("연도")
      .size()
      .rename("음수 관측치 수")
      .reset_index()
)

print("\n[업종별 영업이익률 음수 관측치 수]")
display(
    df[df["영업이익률"] < 0]
      .groupby("업종")
      .size()
      .sort_values(ascending=False)
      .rename("음수 관측치 수")
      .reset_index()
)

,변수,음수 관측치 수,음수 비율(%),관련 업종 수,최초 연도,최종 연도
0,영업이익,51,3.98,17,1995,2025
1,당기순이익,151,11.80,34,1994,2025
2,영업이익률,52,4.06,17,1995,2025
3,이자보상비율,52,4.06,17,1995,2025



[연도별 영업이익률 음수 관측치 수]


,연도,음수 관측치 수
0,1995,1
1,1996,1
2,1997,1
3,1998,2
4,1999,3
5,2000,2
6,2001,2
7,2002,1
8,2003,2
9,2004,2



[업종별 영업이익률 음수 관측치 수]


,업종,음수 관측치 수
0,기타 수송장비,8
1,철도,7
2,조선,7
3,전지,6
4,반도체,6
5,디스플레이,5
6,항공,2
7,석유정제,2
8,고무,1
9,자동차,1


### 외부 자료와의 대조

음수 관측치가 집중된 주요 시기·업종을 공식 자료와 대조한 결과,  
다음과 같이 실제 경기·산업 여건 악화와 일치하는 사례가 확인된다.

| 데이터에서 확인된 구간 | 외부 근거 | 해석 |
|---|---|---|
| **1997~1998년** 다수 업종에서 당기순이익 음수 | 한국은행의 「1998년 기업경영분석 결과」는 1998년 제조업에서 환율 상승에 따른 재료비 증가, 감가상각비·대손상각비 증가 등으로 영업이익이 줄고 경상이익 적자폭이 확대되는 등 수익성이 크게 악화되었다고 설명한다. | 외환위기 전후 여러 제조업에서 음의 순이익이 동시에 나타나는 것은 당시 제조업 전반의 수익성 악화와 방향이 일치한다. |
| **2001~2003년 반도체** 영업이익률 및 이자보상비율 음수 | KDI는 2001년 세계 IT 경기 위축과 반도체 가격 폭락으로 국내 반도체 생산과 수출이 크게 감소했으며, 조속한 회복을 기대하기 어려운 상황이라고 평가하였다. | 반도체 업종의 2001~2003년 음수 수익성 값은 당시 반도체 경기 침체와 경제적으로 부합한다. |
| **2013~2016년 조선** 영업이익·영업이익률 음수 반복 | 산업연구원은 2015년 국내 조선해양플랜트 업계가 신규 발주 감소, 유가 하락, 해양플랜트 공사 지연·취소에 따른 비용 증가로 **대규모 적자**를 기록했다고 평가하였다. | 조선업의 2013~2016년 반복적인 음수 영업수익성은 당시 산업 구조조정 및 대규모 손실과 방향이 일치한다. |
| **2019년** 디스플레이·컴퓨터 등 일부 IT 업종의 음수 수익성 | 한국은행은 2019년 법인기업의 성장성·수익성·안정성이 전년보다 악화되었고, 매출액영업이익률이 6.9%에서 4.7%로 하락했다고 발표하였다. 또한 반도체·LCD 등 주력 수출품 가격 하락을 주요 가격요인으로 설명하였다. | 2019년 일부 IT 업종의 음수 수익성은 당시 전반적 기업 수익성 하락 및 IT 제품 가격 약세와 일치한다. |
| **2020년 석유정제** 영업이익률 음수 | 한국은행은 코로나19 확산으로 글로벌 수요가 위축되면서 석유화학·정제 등의 수출이 감소했다고 보고하였다. 또한 2020년 글로벌 에너지 소비가 코로나19 충격으로 전년 대비 크게 감소했다고 분석하였다. | 2020년 석유정제 업종의 영업손실은 코로나19에 따른 석유수요·수출 부진과 경제적으로 부합한다. |
| **2021~2022년 조선** 영업손실 지속 | 한국은행은 2021년 후판가격 급등이 조선사 수익성에 부정적 요인으로 작용할 가능성을 지적했고, 2022년에도 수주 호황에도 불구하고 조선업 기업의 매출 및 영업손익이 여전히 저조하다고 평가하였다. | 수주 회복과 실제 손익 개선 사이에 시차가 존재할 수 있으며, 2021~2022년 조선업 음수 수익성은 공식 산업진단과 부합한다. |
| **2023년 반도체 등 IT 업종** 음수 수익성 | 한국은행은 2023년 법인기업의 매출액영업이익률이 5.3%에서 3.8%로 하락했다고 발표하였다. 산업연구원도 2023년 글로벌 수요 부진과 IT제품 시장 위축으로 반도체·디스플레이 등의 실적이 부진했다고 평가하였다. | 2023년 반도체·디스플레이 등에서 나타난 음수 영업수익성은 당시 IT 경기 부진과 방향이 일치한다. |
| **2024~2025년 전지** 영업이익률 음수 | 산업연구원은 2024년 배터리 업황 악화의 핵심 원인으로 전기차 시장 성장 둔화를 제시했고, 한국은행도 2024년 이차전지 업황 부진과 생산·수출 약화를 언급하였다. | 전지 업종의 최근 음수 수익성은 전기차 수요 둔화와 배터리 업황 악화라는 산업 여건과 일치한다. |

### 참고한 공식 자료

1. 한국은행, **1998년 기업경영분석 결과**  
   https://www.bok.or.kr/portal/bbs/P0000551/view.do?menuNo=200484&nttId=5837

2. KDI, **반도체 산업이 경기에 미치는 영향** (2001)  
   https://www.kdi.re.kr/share/pressView?bd_no=192

3. 산업연구원(KIET), **조선해양플랜트 산업의 회고와 전망** (2016)  
   https://www.kiet.re.kr/research/economyDetailView?detail_no=1499

4. 한국은행, **2019년 외감기업 경영분석(속보)**  
   https://www.bok.or.kr/portal/bbs/B0000501/view.do?menuNo=201264&nttId=10058574

5. 한국은행, **2019년 4/4분기 및 연간 국민소득(잠정)**  
   https://www.bok.or.kr/portal/bbs/B0000170/view.do?menuNo=200060&nttId=10056865

6. 한국은행, **지역경제보고서(2020년 3월)**  
   https://www.bok.or.kr/portal/bbs/P0002507/view.do?menuNo=200444&nttId=10057287

7. 한국은행, **글로벌 에너지시장의 최근 상황 점검**  
   https://www.bok.or.kr/portal/bbs/P0000528/view.do?menuNo=200434&nttId=10067538

8. 한국은행, **최근 조선업 현황 및 경남 지역경제에 미치는 영향**  
   https://www.bok.or.kr/portal/bbs/P0000720/view.do?menuNo=200570&nttId=10071981&searchBbsSeCd=z22

9. 한국은행, **2023년 기업경영분석 결과(속보)**  
   https://www.bok.or.kr/portal/bbs/B0000501/view.do?menuNo=201263&nttId=10084589&programType=newsData&relate=Y

10. 산업연구원(KIET), **2024년 경제·산업 전망**  
    https://d.kiet.re.kr/trends/ecolookView?ecolook_no=46

11. 산업연구원(KIET), **최근 배터리 업황 긴급 진단: 부진 원인과 향후 전망을 중심으로** (2024)  
    https://www.kiet.re.kr/research/economyDetailView?detail_no=2996

### 판단

외부 자료를 확인한 결과, 데이터에서 반복적으로 나타나는 주요 음수 수익성 관측치는  
**동일 시기의 실제 경기침체·산업 부진·원가 상승·수요 감소와 대체로 방향이 일치한다.**

따라서 음수값을 단순한 입력오류나 제거 대상 이상치로 간주하는 것은 적절하지 않다.  
원자료에서는 그대로 유지하고, 최종 분석기간에 포함되는 음수 관측치는 회귀분석 전 다시 확인한다.

다만 위 자료들은 **각 업종·연도의 음수 관측치가 정확히 해당 요인 하나 때문에 발생했다는 것을 개별적으로 입증하는 자료는 아니다.**  
여기서는 해당 음수값이 현실적으로 발생 가능한 값인지 확인하는 **외부 타당성 검증 근거**로 사용한다.

## 6. 주요 수익성 지표 산식 검증

현재 파일에서 직접 검산할 수 있는 주요 비율은 `영업이익률`이다.

일반적인 형태인

**영업이익률 = 영업이익 / 매출액 × 100**

으로 다시 계산하여 원자료 값과 비교한다.

`이자보상비율`은 필요한 `이자비용` 변수가 수익성 파일에 포함되어 있지 않으므로  
현재 파일만으로는 독립적인 산식 검증을 수행하지 않는다.

In [8]:
formula_check = df.copy()

formula_check["영업이익률_calc"] = (
    formula_check["영업이익"]
    / formula_check["매출액"]
    * 100
)

formula_check["영업이익률_절대오차"] = (
    formula_check["영업이익률"]
    - formula_check["영업이익률_calc"]
).abs()

error_summary = pd.DataFrame({
    "항목": [
        "평균 절대오차",
        "중앙값 절대오차",
        "95% 분위 절대오차",
        "최대 절대오차",
        "절대오차 > 0.5%p 관측치"
    ],
    "값": [
        formula_check["영업이익률_절대오차"].mean(),
        formula_check["영업이익률_절대오차"].median(),
        formula_check["영업이익률_절대오차"].quantile(0.95),
        formula_check["영업이익률_절대오차"].max(),
        (formula_check["영업이익률_절대오차"] > 0.5).sum()
    ]
})

display(error_summary.round(4))

print("\n[영업이익률 재계산 오차가 큰 관측치]")
display(
    formula_check.nlargest(10, "영업이익률_절대오차")[
        [
            "업종코드", "업종", "연도",
            "매출액", "영업이익",
            "영업이익률", "영업이익률_calc",
            "영업이익률_절대오차"
        ]
    ].reset_index(drop=True).round(4)
)

,항목,값
0,평균 절대오차,0.0369
1,중앙값 절대오차,0.0266
2,95% 분위 절대오차,0.0626
3,최대 절대오차,3.2000
4,절대오차 > 0.5%p 관측치,8.0000



[영업이익률 재계산 오차가 큰 관측치]


,업종코드,업종,연도,매출액,영업이익,영업이익률,영업이익률_calc,영업이익률_절대오차
0,I3104,컴퓨터,1994,11.0,0.0,3.2,0.0000,3.2000
1,I3208,철도,1994,26.0,3.0,9.9,11.5385,1.6385
2,I3208,철도,1996,49.0,3.0,5.2,6.1224,0.9224
3,I3208,철도,1998,60.0,2.0,2.5,3.3333,0.8333
4,I3104,컴퓨터,1996,45.0,1.0,2.9,2.2222,0.6778
5,I3104,컴퓨터,1998,68.0,2.0,3.6,2.9412,0.6588
6,I3209,기타 수송장비,1995,81.0,-5.0,-6.8,-6.1728,0.6272
7,I3104,컴퓨터,1997,68.0,2.0,2.4,2.9412,0.5412
8,I3209,기타 수송장비,1994,91.0,1.0,0.8,1.0989,0.2989
9,I3104,컴퓨터,1999,173.0,10.0,5.5,5.7803,0.2803


### 해석

`영업이익률`을 `영업이익 / 매출액 × 100`으로 재계산한 결과  
대부분의 관측치에서 원자료와의 차이는 매우 작게 나타난다.

다만 매출액이나 영업이익의 절대 규모가 매우 작은 일부 과거 관측치에서는 오차가 상대적으로 크게 나타난다.  
이는 원자료의 금액 변수가 표시 단위에서 반올림된 반면, 비율은 더 세밀한 원수치로 계산되었을 가능성이 있으므로  
현재 단계에서 단순 입력오류로 판단하지 않는다.

따라서 `영업이익률`은 전반적으로 금액 변수와 내부적으로 일관적인 것으로 본다.

`이자보상비율`은 이 파일에 `이자비용`이 없기 때문에 여기서는 산식 검증을 보류하며,  
안정성 자료 또는 최종 병합자료와 함께 확인하는 것이 적절하다.

## 7. 원자료 검증 결론

### 최종 판단

수익성 원자료는 다음 조건을 충족한다.

- 40개 업종 × 32년의 완전 균형패널
- 업종코드와 업종명의 1:1 대응
- 업종-연도 중복 없음
- 결측치 없음
- 무한대값 없음
- 매출액 전 관측치 양수
- 영업이익률과 금액 변수 간 전반적인 내부 산식 일관성 확인

영업이익·당기순이익·영업이익률·이자보상비율에서 음수값이 존재하지만,  
공식 외부자료를 대조한 결과 주요 음수 구간은 실제 경기침체·산업 부진·원가 상승·수요 감소가 있었던 시기와 대체로 일치한다.  
따라서 이를 단순 입력오류로 보지 않고 원자료에 유지한다.

또한 이자보상비율에는 큰 양의 극단값이 존재하지만  
현재 파일만으로는 이자비용을 확인할 수 없어 해당 값을 오류로 단정하지 않는다.

따라서 **수익성 원자료는 후속 분석에 사용 가능한 것으로 판단한다.**

---

### 이후 분석 원칙

이 원자료에서는 여기까지 검증하고 종료한다.

다음과 같은 심화 EDA는 **최종 분석용 병합패널**에서 수행한다.

- 최종 분석기간의 수익성 지표 분포
- 적자 및 극단 관측치의 최종 분석표본 포함 여부
- 업종별·연도별 수익성 변화
- 이자보상비율의 분석기간 내 극단값 검토
- 원자재 노출도와 수익성 지표의 관계
- 평행추세 및 DiD 식별 점검